# 🎓 Chatbot Tư Vấn Tuyển Sinh FPT School

**RAG Chatbot** sử dụng:
- 🤖 LLM: `Qwen/Qwen2.5-1.5B-Instruct`
- 🔍 Embedding: `intfloat/multilingual-e5-small`
- 🗄️ Vector DB: FAISS
- 🖥️ UI: Gradio

---
> ⚡ **Khuyến nghị**: Chạy với **GPU T4** (Runtime → Change runtime type → T4 GPU)

## 🔧 Bước 1: Kiểm tra GPU & môi trường

In [1]:
import torch

print(f'PyTorch version: {torch.__version__}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Không có GPU. Chạy bằng CPU sẽ chậm hơn (~3–5 phút/câu)')
    print('   → Vào Runtime > Change runtime type > chọn T4 GPU')

PyTorch version: 2.11.0+cu128
✅ GPU: Tesla T4
   VRAM: 15.6 GB


## 📦 Bước 2: Clone repository từ GitHub

In [2]:
import os

REPO_URL = 'https://github.com/Luanntc2/chatbot-tuyen-sinh.git'
PROJECT_DIR = '/content/chatbot-tuyen-sinh'

if os.path.exists(PROJECT_DIR):
    print('🔄 Repo đã tồn tại, pulling updates...')
    os.system(f'git -C {PROJECT_DIR} pull')
else:
    print('📥 Đang clone repository...')
    os.system(f'git clone {REPO_URL} {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
print(f'\n✅ Working directory: {os.getcwd()}')
os.system('ls -la')

📥 Đang clone repository...

✅ Working directory: /content/chatbot-tuyen-sinh


0

## Bước 3: Cài đặt dependencies

> ⏳ Bước này mất khoảng **3–5 phút**

In [3]:
import os, sys, torch

os.chdir('/content/chatbot-tuyen-sinh')

# Chọn faiss-gpu hoặc faiss-cpu tùy môi trường
faiss_pkg = 'faiss-gpu' if torch.cuda.is_available() else 'faiss-cpu>=1.7.4'
print(f'📦 Sẽ cài: {faiss_pkg}')

# Đọc requirements, loại dòng faiss
with open('requirements.txt') as f:
    lines = [l for l in f if not l.strip().startswith('faiss') and not l.strip().startswith('#')]
reqs = ''.join(lines)

with open('/tmp/req_colab.txt', 'w') as f:
    f.write(reqs)

os.system(f'pip install -q {faiss_pkg}')
os.system('pip install -q -r /tmp/req_colab.txt')
print('\n✅ Cài xong tất cả dependencies!')

📦 Sẽ cài: faiss-gpu

✅ Cài xong tất cả dependencies!


## 🗄️ Bước 4: Build FAISS Vector Index

Nếu index đã có trong repo, bước này bỏ qua tự động.

In [4]:
import os, sys
sys.path.insert(0, '/content/chatbot-tuyen-sinh')
os.chdir('/content/chatbot-tuyen-sinh')

from config import FAISS_INDEX_PATH

if os.path.exists(os.path.join(FAISS_INDEX_PATH, 'index.faiss')):
    print(f'✅ FAISS index đã tồn tại: {FAISS_INDEX_PATH}')
    print('   Bỏ qua bước build.')
else:
    print('🔄 Đang build FAISS index từ dữ liệu...')
    from src.ingest import ingest
    ingest()
    print('✅ Build FAISS index hoàn tất!')

🔄 Đang build FAISS index từ dữ liệu...
=== Bắt đầu ingest dữ liệu từ 'data/raw' ===

1. Đọc tài liệu...
  [OK] chuong_trinh_hoc.txt — 1 document(s)
  [OK] hoc_phi_chi_tiet.txt — 1 document(s)
  [OK] tuyen_sinh_fpt.txt — 1 document(s)

   Tổng: 3 file (3 .txt, 0 .pdf) → 3 document(s)

2. Chia chunk...
   Tạo ra 45 chunk(s)

3. Tạo embeddings và lưu FAISS index...
  Đang tải embedding model (lần đầu sẽ tải về từ HuggingFace)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

  Đang tạo FAISS index...
   Đã lưu vào 'vectorstore/faiss_index'

=== Thống kê ===
  Số file đọc   : 3
  Số chunk tạo  : 45
  FAISS index   : vectorstore/faiss_index

Ingest hoàn thành!
✅ Build FAISS index hoàn tất!


## 🤖 Bước 5: Load RAG Pipeline

In [5]:
import os, sys
sys.path.insert(0, '/content/chatbot-tuyen-sinh')
os.chdir('/content/chatbot-tuyen-sinh')

print('🔄 Đang load pipeline (lần đầu mất 2–3 phút để download model)...')
from src.pipeline import RAGPipeline

pipeline = RAGPipeline()
print('\n✅ Pipeline sẵn sàng!')

🔄 Đang load pipeline (lần đầu mất 2–3 phút để download model)...
=== Khởi động RAG Pipeline ===

  Tải embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Tải FAISS index từ 'vectorstore/faiss_index'...
  Retriever sẵn sàng.

  Tải model: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

...

  GPU: Tesla T4 (15.6 GB VRAM)
  Dùng float16


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  Model 'Qwen/Qwen2.5-1.5B-Instruct' sẵn sàng.

=== Pipeline sẵn sàng ===


✅ Pipeline sẵn sàng!


## 🧪 Bước 6 (Tùy chọn): Test nhanh CLI

In [6]:
test_question = 'Học phí FPT School là bao nhiêu?'

print(f'❓ Câu hỏi: {test_question}')
print('─' * 60)
print('💬 Câu trả lời:')

for kind, value in pipeline.stream(test_question):
    if kind == 'token':
        print(value, end='', flush=True)
    elif kind == 'answer':
        print(value, end='', flush=True)

print('\n' + '─' * 60)

❓ Câu hỏi: Học phí FPT School là bao nhiêu?
────────────────────────────────────────────────────────────
💬 Câu trả lời:
  [Pipeline] Câu hỏi TRONG phạm vi → streaming...
Theo thông tin được cung cấp, học phí FPT School được tính theo đơn vị năm học và chia thành 2 lần thu trước mỗi học kỳ. Mỗi học kỳ, học phí sẽ được chia làm hai phần:

1. Phí xét tuyển/Phí thi học bổng: 200.000 VNĐ.
2. Phí nhập học: 2.000.000 VNĐ.

Vì vậy, tổng học phí cho một học kỳ là:
200.000 VNĐ + 2.000.000 VNĐ = 2.200.000 VNĐ.

Do đó, học phí FPT School cho một học kỳ là 2.200.000 VNĐ.
────────────────────────────────────────────────────────────


## 🚀 Bước 7: Chạy Gradio Web UI

Gradio sẽ tạo **public link** (dạng `https://xxxx.gradio.live`) để truy cập từ bất kỳ đâu.

> ⏱️ Link có hiệu lực trong **72 giờ**

In [7]:
import os, sys
sys.path.insert(0, '/content/chatbot-tuyen-sinh')
os.chdir('/content/chatbot-tuyen-sinh')

import app as chatbot_app

# Inject pipeline đã load (tránh load lại)
chatbot_app.pipeline = pipeline

print('🚀 Khởi động Gradio...')
print('📌 Click vào link "Running on public URL" bên dưới để mở chatbot')
print('─' * 60)

chatbot_app.demo.launch(
    share=True,
    debug=False,
    show_error=True
)

/content/chatbot-tuyen-sinh/app.py:217: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_ui = gr.Chatbot(


🚀 Khởi động Gradio...
📌 Click vào link "Running on public URL" bên dưới để mở chatbot
────────────────────────────────────────────────────────────
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a6e7956bce41b8aa6b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

## 🛠️ Xử lý sự cố

### ❌ CUDA out of memory
```python
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# Chạy dòng trên rồi Restart & Run All
```

### ❌ ModuleNotFoundError
```bash
!pip install -q <tên_module>
# Sau đó Runtime → Restart session → chạy lại từ Bước 3
```

### ❌ Gradio không tạo được public URL
```python
# Thay share=True bằng share=False để xem trong Colab output
chatbot_app.demo.launch(share=False)
```

### 🔁 Reset hoàn toàn
Runtime → Disconnect and delete runtime → chạy lại từ đầu.

## 🔄 (Tùy chọn) Rebuild FAISS index khi có dữ liệu mới

In [8]:
# ⚠️ Chỉ chạy khi đã thêm file mới vào data/raw/
import shutil, os, sys
sys.path.insert(0, '/content/chatbot-tuyen-sinh')
os.chdir('/content/chatbot-tuyen-sinh')
from config import FAISS_INDEX_PATH

if os.path.exists(FAISS_INDEX_PATH):
    shutil.rmtree(FAISS_INDEX_PATH)
    print(f'🗑️ Đã xóa index cũ')

from src.ingest import ingest
ingest()
print('✅ Rebuild hoàn tất!')

🗑️ Đã xóa index cũ
=== Bắt đầu ingest dữ liệu từ 'data/raw' ===

1. Đọc tài liệu...
  [OK] chuong_trinh_hoc.txt — 1 document(s)
  [OK] hoc_phi_chi_tiet.txt — 1 document(s)
  [OK] tuyen_sinh_fpt.txt — 1 document(s)

   Tổng: 3 file (3 .txt, 0 .pdf) → 3 document(s)

2. Chia chunk...
   Tạo ra 45 chunk(s)

3. Tạo embeddings và lưu FAISS index...
  Đang tải embedding model (lần đầu sẽ tải về từ HuggingFace)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Đang tạo FAISS index...
   Đã lưu vào 'vectorstore/faiss_index'

=== Thống kê ===
  Số file đọc   : 3
  Số chunk tạo  : 45
  FAISS index   : vectorstore/faiss_index

Ingest hoàn thành!
✅ Rebuild hoàn tất!
